# 09i ResNet101-IBN ArcFace Training

This notebook trains a CityFlowV2 vehicle ReID model with the improved ResNet101-IBN-a recipe:

- ArcFace head on BNNeck features
- Triplet loss on pre-BN global features
- Center loss delayed until epoch 20
- MultiStep LR with warmup
- Market-1501 evaluation plus re-ranking reference metrics

The notebook is self-contained and reuses the 09d crop extraction and split-building flow.

In [ ]:
import os
import sys
import shutil
import subprocess


def run_pip(args):
    command = [sys.executable, "-m", "pip", "install"] + args
    print("[bootstrap] Running:", " ".join(command))
    subprocess.check_call(command)


def detect_gpu_compute_caps():
    nvidia_smi = shutil.which("nvidia-smi")
    if not nvidia_smi:
        print("[bootstrap] nvidia-smi not found; skipping GPU capability detection")
        return []

    result = subprocess.run(
        [
            nvidia_smi,
            "--query-gpu=gpu_name,compute_cap",
            "--format=csv,noheader",
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("[bootstrap] GPU detection failed:")
        print(result.stderr.strip() or result.stdout.strip())
        return []

    entries = []
    for raw_line in result.stdout.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        name, _, cap_text = line.partition(",")
        try:
            capability = float(cap_text.strip())
        except ValueError:
            print(f"[bootstrap] Could not parse compute capability from: {line}")
            continue
        entries.append((name.strip(), capability))
    return entries


gpu_entries = detect_gpu_compute_caps()
if gpu_entries:
    for gpu_name, capability in gpu_entries:
        print(f"[bootstrap] Detected GPU: {gpu_name} (compute capability {capability:.1f})")
else:
    print("[bootstrap] No GPU capability entries detected")


needs_p100_compatible_torch = any(capability < 7.0 for _, capability in gpu_entries)
if needs_p100_compatible_torch:
    print("[bootstrap] Detected pre-Volta GPU; installing PyTorch 2.4.1/cu124 for P100 compatibility")
    run_pip([
        "torch==2.4.1",
        "torchvision==0.19.1",
        "--index-url",
        "https://download.pytorch.org/whl/cu124",
    ])
else:
    print("[bootstrap] Torch downgrade not required")


print("[bootstrap] Installing notebook dependencies")
run_pip([
    "loguru",
    "matplotlib",
    "numpy",
    "opencv-python-headless",
    "Pillow",
    "scipy",
])
print("[bootstrap] Environment setup complete")

In [ ]:
from pathlib import Path


INPUT_ROOT = Path("/kaggle/input")
WEIGHTS_ROOT = INPUT_ROOT / "mtmc-weights"
DATA_ROOT = INPUT_ROOT / "data-aicity-2023-track-2"


def print_tree(root, max_depth=3):
    if not root.exists():
        print(f"  <missing {root}>")
        return
    print(f"  {root}/")
    for path in sorted(root.rglob("*"), key=lambda item: str(item)):
        rel = path.relative_to(root)
        if len(rel.parts) > max_depth:
            continue
        indent = "    " * (len(rel.parts) - 1)
        suffix = "/" if path.is_dir() else ""
        print(f"{indent}  - {rel}{suffix}")


print("[debug] Listing /kaggle/input (depth<=3)")
print_tree(INPUT_ROOT, max_depth=3)
print("[debug] Listing /kaggle/input/mtmc-weights (depth<=3)")
print_tree(WEIGHTS_ROOT, max_depth=3)
print("[debug] Listing /kaggle/input/data-aicity-2023-track-2 (depth<=3)")
print_tree(DATA_ROOT, max_depth=3)

In [ ]:
import os


script_content = r'''#!/usr/bin/env python3
# 09i ResNet101-IBN-a ArcFace training for CityFlowV2.

import atexit
import gc
import json
import math
import os
import shutil
import socket
import sys
import time
import traceback
import urllib.request
from collections import defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
import torchvision.transforms as T
from PIL import Image
from torch.optim.lr_scheduler import LinearLR, MultiStepLR, SequentialLR
from torch.utils.data import DataLoader, Dataset, Sampler


LOG_PATH = Path("/kaggle/working/debug_09i.log")
LOG_HANDLE = open(LOG_PATH, "w", buffering=1)


class TeeWriter:
    def __init__(self, original, log_handle):
        self.original = original
        self.log_handle = log_handle

    def write(self, message):
        self.original.write(message)
        try:
            self.log_handle.write(message)
        except Exception:
            pass

    def flush(self):
        self.original.flush()
        try:
            self.log_handle.flush()
        except Exception:
            pass


def dedupe_paths(paths):
    unique = []
    seen = set()
    for path in paths:
        text = str(path)
        if text in seen:
            continue
        seen.add(text)
        unique.append(path)
    return unique


def find_named_paths(root: Path, name: str, want_dir=None, max_results=50):
    if not root.exists():
        return []
    matches = []
    for path in root.rglob(name):
        if want_dir is True and not path.is_dir():
            continue
        if want_dir is False and not path.is_file():
            continue
        matches.append(path)
        if len(matches) >= max_results:
            break
    return sorted(matches)


def resolve_checkpoint_path(kaggle_input_root: Path, expected_path: str):
    candidate = Path(expected_path)
    if candidate.exists():
        print(f"[debug] Found checkpoint at expected path: {candidate}")
        return str(candidate)
    matches = find_named_paths(kaggle_input_root, candidate.name, want_dir=False, max_results=20)
    if matches:
        print(f"[debug] Resolved checkpoint {candidate.name}: {matches[0]}")
        return str(matches[0])
    return None


def resolve_cityflow_train_root(kaggle_input_root: Path):
    expected_dataset_root = kaggle_input_root / "data-aicity-2023-track-2"
    candidate_bases = dedupe_paths(
        [expected_dataset_root]
        + find_named_paths(kaggle_input_root, "data-aicity-2023-track-2", want_dir=True, max_results=20)
    )
    candidate_train_roots = []
    for base in candidate_bases:
        for subpath in ("AIC22_Track1_MTMC_Tracking/train", "train"):
            candidate = base / subpath
            if candidate.exists():
                candidate_train_roots.append(candidate)
    candidate_train_roots = dedupe_paths(candidate_train_roots)
    if candidate_train_roots:
        best_root = max(candidate_train_roots, key=lambda root: (len(list(root.glob("S*/c*"))), str(root)))
        print(f"[debug] Resolved CityFlowV2 train root: {best_root}")
        return best_root

    inferred_roots = []
    for gt_path in find_named_paths(kaggle_input_root, "gt.txt", want_dir=False, max_results=200):
        if gt_path.parent.name != "gt":
            continue
        camera_dir = gt_path.parent.parent
        scene_dir = camera_dir.parent
        if not camera_dir.name.startswith("c") or not scene_dir.name.startswith("S"):
            continue
        video_file = camera_dir / "vdo.avi"
        if not video_file.exists():
            continue
        inferred_roots.append(scene_dir.parent)
    inferred_roots = dedupe_paths(inferred_roots)
    if inferred_roots:
        best_root = max(inferred_roots, key=lambda root: (len(list(root.glob("S*/c*"))), str(root)))
        print(f"[debug] Inferred CityFlowV2 train root: {best_root}")
        return best_root
    raise FileNotFoundError("CityFlowV2 dataset not found anywhere under /kaggle/input")


def normalize_numpy(features: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    norms = np.clip(norms, a_min=1e-12, a_max=None)
    return features / norms


def re_ranking(query_features, gallery_features, k1=20, k2=6, lambda_value=0.3):
    query_features = normalize_numpy(query_features.astype(np.float32))
    gallery_features = normalize_numpy(gallery_features.astype(np.float32))
    all_features = np.concatenate([query_features, gallery_features], axis=0)
    all_num = all_features.shape[0]
    query_num = query_features.shape[0]
    original_dist = 2.0 - 2.0 * np.matmul(all_features, all_features.T)
    original_dist = np.maximum(original_dist, 0.0)
    initial_rank = np.argsort(original_dist, axis=1).astype(np.int32)
    V = np.zeros_like(original_dist, dtype=np.float32)

    for index in range(all_num):
        forward_neighbors = initial_rank[index, : k1 + 1]
        backward_neighbors = initial_rank[forward_neighbors, : k1 + 1]
        reciprocal_mask = np.any(backward_neighbors == index, axis=1)
        reciprocal_neighbors = forward_neighbors[reciprocal_mask]
        reciprocal_expansion = reciprocal_neighbors.copy()
        for candidate in reciprocal_neighbors:
            candidate_forward = initial_rank[candidate, : int(np.around(k1 / 2)) + 1]
            candidate_backward = initial_rank[candidate_forward, : int(np.around(k1 / 2)) + 1]
            candidate_mask = np.any(candidate_backward == candidate, axis=1)
            candidate_reciprocal = candidate_forward[candidate_mask]
            if candidate_reciprocal.size == 0:
                continue
            overlap = np.intersect1d(candidate_reciprocal, reciprocal_neighbors)
            if overlap.size > (2.0 / 3.0) * candidate_reciprocal.size:
                reciprocal_expansion = np.append(reciprocal_expansion, candidate_reciprocal)
        reciprocal_expansion = np.unique(reciprocal_expansion)
        weights = np.exp(-original_dist[index, reciprocal_expansion])
        V[index, reciprocal_expansion] = weights / np.sum(weights)

    if k2 > 1:
        V_qe = np.zeros_like(V, dtype=np.float32)
        for index in range(all_num):
            V_qe[index, :] = np.mean(V[initial_rank[index, :k2], :], axis=0)
        V = V_qe

    inverse_index = [np.where(V[:, column] != 0)[0] for column in range(all_num)]
    jaccard_dist = np.zeros((query_num, all_num), dtype=np.float32)
    for index in range(query_num):
        temp_min = np.zeros((1, all_num), dtype=np.float32)
        non_zero = np.where(V[index, :] != 0)[0]
        for column in non_zero:
            related = inverse_index[column]
            temp_min[0, related] += np.minimum(V[index, column], V[related, column])
        jaccard_dist[index] = 1.0 - temp_min / (2.0 - temp_min)

    final_dist = jaccard_dist * (1.0 - lambda_value) + original_dist[:query_num, :] * lambda_value
    return final_dist[:, query_num:]


class ReIDDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        path, pid, cam = self.data[index]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, pid, cam, path


class PKSampler(Sampler):
    def __init__(self, data, p, k):
        self.p = p
        self.k = k
        self.pid_to_indices = defaultdict(list)
        for index, (_, pid, _) in enumerate(data):
            self.pid_to_indices[pid].append(index)
        self.pids = list(self.pid_to_indices.keys())
        self.batch_size = p * k

    def __iter__(self):
        pids = list(self.pids)
        np.random.shuffle(pids)
        batch = []
        for pid in pids:
            indices = self.pid_to_indices[pid]
            replace = len(indices) < self.k
            selected = np.random.choice(indices, size=self.k, replace=replace).tolist()
            batch.extend(selected)
            if len(batch) >= self.batch_size:
                yield from batch[: self.batch_size]
                batch = batch[self.batch_size :]
        if batch:
            yield from batch

    def __len__(self):
        return len(self.pids) * self.k


class IBN_a(nn.Module):
    def __init__(self, planes):
        super().__init__()
        half = planes // 2
        self.IN = nn.InstanceNorm2d(half, affine=True)
        self.BN = nn.BatchNorm2d(planes - half)

    def forward(self, x):
        split = x.shape[1] // 2
        return torch.cat([self.IN(x[:, :split]), self.BN(x[:, split:])], dim=1)


class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p), 1).pow(1.0 / self.p)


class ArcFaceHead(nn.Module):
    # Additive angular margin head for vehicle re-identification.

    def __init__(self, in_features, num_classes, s=30.0, m=0.35):
        super().__init__()
        self.s = s
        self.m = m
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m
        self.weight = nn.Parameter(torch.empty(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, features, labels=None):
        cosine = F.linear(F.normalize(features), F.normalize(self.weight))
        cosine = cosine.clamp(-1.0 + 1e-7, 1.0 - 1e-7)
        if labels is None:
            return cosine * self.s
        sine = torch.sqrt((1.0 - torch.pow(cosine, 2)).clamp(0.0, 1.0))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1.0)
        output = one_hot * phi + (1.0 - one_hot) * cosine
        return output * self.s


class CrossEntropyLabelSmooth(nn.Module):
    def __init__(self, num_classes, epsilon=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, inputs, targets):
        log_probs = self.logsoftmax(inputs)
        targets_one_hot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
        targets_smooth = (1.0 - self.epsilon) * targets_one_hot + self.epsilon / self.num_classes
        return (-targets_smooth * log_probs).mean(0).sum()


class TripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, inputs, targets):
        n = inputs.size(0)
        dist = torch.pow(inputs, 2).sum(dim=1, keepdim=True).expand(n, n)
        dist = dist + dist.t()
        dist.addmm_(inputs, inputs.t(), beta=1, alpha=-2)
        dist = dist.clamp(min=1e-12).sqrt()
        mask = targets.expand(n, n).eq(targets.expand(n, n).t())
        dist_ap = []
        dist_an = []
        for i in range(n):
            dist_ap.append(dist[i][mask[i]].max().unsqueeze(0))
            dist_an.append(dist[i][~mask[i]].min().unsqueeze(0))
        dist_ap = torch.cat(dist_ap)
        dist_an = torch.cat(dist_an)
        y = torch.ones_like(dist_an)
        return self.ranking_loss(dist_an, dist_ap, y)


class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim):
        super().__init__()
        self.num_classes = num_classes
        self.feat_dim = feat_dim
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim))

    def forward(self, features, labels):
        batch_size = features.size(0)
        distmat = (
            torch.pow(features, 2).sum(dim=1, keepdim=True).expand(batch_size, self.num_classes)
            + torch.pow(self.centers, 2).sum(dim=1, keepdim=True).expand(self.num_classes, batch_size).t()
        )
        distmat.addmm_(features, self.centers.t(), beta=1, alpha=-2)
        classes = torch.arange(self.num_classes, device=features.device).long()
        labels = labels.unsqueeze(1).expand(batch_size, self.num_classes)
        mask = labels.eq(classes.expand(batch_size, self.num_classes))
        dist = distmat * mask.float()
        return dist.clamp(min=1e-12, max=1e12).sum() / batch_size


IBN_NET_RESNET101_A_URL = "https://github.com/XingangPan/IBN-Net/releases/download/v1.0/resnet101_ibn_a-59ea0ac6.pth"
IBN_NET_RESNET101_A_LOCAL_PATH = "/kaggle/working/resnet101_ibn_a.pth"


class ReIDModelResNet101IBNArcFace(nn.Module):
    def __init__(self, num_classes, feat_dim=2048, last_stride=1, gem_p=3.0, arcface_s=30.0, arcface_m=0.35, pretrained=True):
        super().__init__()
        base = tv_models.resnet101(weights=None)
        for layer in [base.layer1, base.layer2, base.layer3]:
            for block in layer:
                if hasattr(block, "bn1"):
                    block.bn1 = IBN_a(block.bn1.num_features)
        if pretrained:
            state_dict = None
            if os.path.exists(IBN_NET_RESNET101_A_LOCAL_PATH):
                state_dict = torch.load(IBN_NET_RESNET101_A_LOCAL_PATH, map_location="cpu", weights_only=False)
            else:
                try:
                    print(f"[debug] Downloading ImageNet weights to {IBN_NET_RESNET101_A_LOCAL_PATH}")
                    original_timeout = socket.getdefaulttimeout()
                    socket.setdefaulttimeout(120)
                    try:
                        urllib.request.urlretrieve(IBN_NET_RESNET101_A_URL, IBN_NET_RESNET101_A_LOCAL_PATH)
                    finally:
                        socket.setdefaulttimeout(original_timeout)
                    state_dict = torch.load(IBN_NET_RESNET101_A_LOCAL_PATH, map_location="cpu", weights_only=False)
                except Exception as error:
                    print(f"[warn] Direct download failed: {error}")
                    state_dict = torch.hub.load_state_dict_from_url(IBN_NET_RESNET101_A_URL, map_location="cpu", progress=True)
            load_result = base.load_state_dict(state_dict, strict=False)
            unexpected_missing = sorted(key for key in load_result.missing_keys if key not in {"fc.weight", "fc.bias"})
            if unexpected_missing:
                raise RuntimeError(f"Unexpected missing ImageNet keys: {unexpected_missing}")
            if load_result.unexpected_keys:
                raise RuntimeError(f"Unexpected ImageNet keys: {load_result.unexpected_keys}")
        if last_stride == 1:
            for module in base.layer4.modules():
                if isinstance(module, nn.Conv2d) and module.stride == (2, 2):
                    module.stride = (1, 1)
        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4
        self.pool = GeM(p=gem_p)
        self.feat_dim = feat_dim
        self.bottleneck = nn.BatchNorm1d(feat_dim)
        self.bottleneck.bias.requires_grad_(False)
        nn.init.constant_(self.bottleneck.weight, 1)
        nn.init.constant_(self.bottleneck.bias, 0)
        self.arcface_head = ArcFaceHead(feat_dim, num_classes, s=arcface_s, m=arcface_m)

    def forward(self, x, labels=None):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        global_feat = self.pool(x).view(x.size(0), -1)
        bn_feat = self.bottleneck(global_feat)
        if self.training:
            cls_score = self.arcface_head(bn_feat, labels)
            return cls_score, global_feat, bn_feat
        return F.normalize(bn_feat, p=2, dim=1)


def unwrap_model(model):
    return model.module if hasattr(model, "module") else model


def get_model_state_dict(model):
    return unwrap_model(model).state_dict()


def load_matching_state_dict(model, state_dict):
    clean_state = {}
    for key, value in state_dict.items():
        clean_key = key[7:] if key.startswith("module.") else key
        if clean_key.startswith("classifier") or clean_key.startswith("arcface_head"):
            continue
        clean_state[clean_key] = value
    model_state = unwrap_model(model).state_dict()
    compatible = {key: value for key, value in clean_state.items() if key in model_state and hasattr(value, "shape") and value.shape == model_state[key].shape}
    result = unwrap_model(model).load_state_dict(compatible, strict=False)
    print(f"[debug] Warm start loaded keys: {len(compatible)} | missing={len(result.missing_keys)} unexpected={len(result.unexpected_keys)}")


def parse_cityflowv2(root):
    train = []
    query = []
    gallery = []
    for split_name, split_list in (("train", train), ("query", query), ("gallery", gallery)):
        split_dir = Path(root) / split_name
        if not split_dir.is_dir():
            raise FileNotFoundError(f"CityFlowV2 ReID split not found: {split_dir}")
        for file_name in sorted(os.listdir(split_dir)):
            if not file_name.endswith(".jpg"):
                continue
            parts = file_name.split("_")
            if len(parts) < 4:
                continue
            pid = int(parts[0])
            cam_name = parts[1] + "_" + parts[2]
            split_list.append((str(split_dir / file_name), pid, cam_name))
    all_cams = sorted({cam for _, _, cam in train + query + gallery})
    cam2id = {cam: index for index, cam in enumerate(all_cams)}
    train = [(path, pid, cam2id[cam]) for path, pid, cam in train]
    query = [(path, pid, cam2id[cam]) for path, pid, cam in query]
    gallery = [(path, pid, cam2id[cam]) for path, pid, cam in gallery]
    train_pids = sorted(set(pid for _, pid, _ in train))
    pid2label = {pid: label for label, pid in enumerate(train_pids)}
    train = [(path, pid2label[pid], cam) for path, pid, cam in train]
    return train, query, gallery, len(train_pids), len(all_cams)


def extract_cityflow_crops(raw_root: Path, reid_root: Path):
    crop_root = Path("/kaggle/working/cityflowv2_crops")
    max_samples_per_track = 15
    min_box_area = 2000
    min_box_side = 30

    if crop_root.exists():
        shutil.rmtree(crop_root)
    crop_root.mkdir(parents=True, exist_ok=True)

    camera_dirs = sorted(path for path in raw_root.glob("S*/c*") if path.is_dir())
    print(f"[debug] Found {len(camera_dirs)} camera directories under {raw_root}")
    all_crops = {}
    total_crop_count = 0

    for cam_dir in camera_dirs:
        scene = cam_dir.parent.name
        cam = cam_dir.name
        cam_id = f"{scene}_{cam}"
        gt_file = cam_dir / "gt" / "gt.txt"
        video_file = cam_dir / "vdo.avi"
        if not gt_file.exists() or not video_file.exists():
            print(f"[warn] Skipping {cam_id}: missing gt.txt or vdo.avi")
            continue

        detections = defaultdict(list)
        with gt_file.open("r", encoding="utf-8") as handle:
            for line in handle:
                parts = line.strip().split(",")
                if len(parts) < 6:
                    continue
                frame_id = int(parts[0])
                track_id = int(parts[1])
                x = int(float(parts[2]))
                y = int(float(parts[3]))
                w = int(float(parts[4]))
                h = int(float(parts[5]))
                if w * h < min_box_area or w < min_box_side or h < min_box_side:
                    continue
                detections[track_id].append((frame_id, x, y, w, h))

        sampled_by_track = {}
        for track_id, dets in detections.items():
            dets.sort(key=lambda item: item[0])
            if len(dets) <= max_samples_per_track:
                sampled = dets
            else:
                sample_indices = np.linspace(0, len(dets) - 1, num=max_samples_per_track, dtype=int)
                sampled = [dets[index] for index in sample_indices]
            sampled_by_track[track_id] = sampled

        dets_by_frame = defaultdict(list)
        for track_id, dets in sampled_by_track.items():
            for frame_id, x, y, w, h in dets:
                dets_by_frame[frame_id].append((track_id, x, y, w, h))

        cap = cv2.VideoCapture(str(video_file))
        if not cap.isOpened():
            print(f"[warn] Skipping {cam_id}: failed to open video")
            continue

        camera_crop_count = 0
        for frame_id in sorted(dets_by_frame.keys()):
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id - 1)
            ok, frame = cap.read()
            if not ok:
                continue
            frame_height, frame_width = frame.shape[:2]
            for track_id, x, y, w, h in dets_by_frame[frame_id]:
                x1 = max(0, x)
                y1 = max(0, y)
                x2 = min(frame_width, x + w)
                y2 = min(frame_height, y + h)
                if x2 <= x1 or y2 <= y1:
                    continue
                crop = frame[y1:y2, x1:x2]
                if crop.size == 0:
                    continue
                file_name = f"{track_id:04d}_{scene}_{cam}_f{frame_id:06d}.jpg"
                crop_path = crop_root / file_name
                if not cv2.imwrite(str(crop_path), crop):
                    continue
                all_crops.setdefault(track_id, {}).setdefault(cam_id, []).append((str(crop_path), frame_id))
                camera_crop_count += 1
        cap.release()
        total_crop_count += camera_crop_count
        print(f"  {cam_id}: {len(detections)} track ids, {camera_crop_count} crops")
        gc.collect()

    if reid_root.exists():
        shutil.rmtree(reid_root)
    for split_name in ("train", "query", "gallery"):
        (reid_root / split_name).mkdir(parents=True, exist_ok=True)

    multi_cam_ids = [track_id for track_id, cams in all_crops.items() if len(cams) >= 2]
    single_cam_ids = [track_id for track_id, cams in all_crops.items() if len(cams) == 1]
    rng = np.random.default_rng(42)
    rng.shuffle(multi_cam_ids)
    n_train = int(len(multi_cam_ids) * 0.7)
    train_ids = set(multi_cam_ids[:n_train])
    eval_ids = set(multi_cam_ids[n_train:])
    counts = {"train": 0, "query": 0, "gallery": 0}

    for track_id in train_ids:
        for items in all_crops[track_id].values():
            for crop_path, frame_id in sorted(items, key=lambda item: item[1]):
                shutil.copy2(crop_path, reid_root / "train" / Path(crop_path).name)
                counts["train"] += 1

    for track_id in eval_ids:
        for items in all_crops[track_id].values():
            ordered_items = sorted(items, key=lambda item: item[1])
            if not ordered_items:
                continue
            query_source, _ = ordered_items[0]
            shutil.copy2(query_source, reid_root / "query" / Path(query_source).name)
            counts["query"] += 1
            for gallery_source, _ in ordered_items[1:]:
                shutil.copy2(gallery_source, reid_root / "gallery" / Path(gallery_source).name)
                counts["gallery"] += 1

    for track_id in single_cam_ids:
        for items in all_crops[track_id].values():
            for crop_path, frame_id in sorted(items, key=lambda item: item[1]):
                shutil.copy2(crop_path, reid_root / "gallery" / Path(crop_path).name)
                counts["gallery"] += 1

    print(f"[debug] Total vehicles: {len(all_crops)} | total crops: {total_crop_count}")
    print(f"[debug] Splits: train={counts['train']}, query={counts['query']}, gallery={counts['gallery']}")
    print(f"[debug] Train IDs: {len(train_ids)}, eval IDs: {len(eval_ids)}, distractor IDs: {len(single_cam_ids)}")


@torch.no_grad()
def extract_features(model, loader, device):
    model.eval()
    features = []
    pids = []
    cams = []
    for images, pid, cam, _ in loader:
        images = images.to(device)
        feats = model(images)
        flip_feats = model(torch.flip(images, dims=[3]))
        avg_feats = F.normalize((feats + flip_feats) / 2.0, p=2, dim=1)
        features.append(avg_feats.cpu().numpy())
        pids.append(pid.numpy())
        cams.append(cam.numpy())
    return np.concatenate(features), np.concatenate(pids), np.concatenate(cams)


def eval_market_style(distmat, query_pids, query_cams, gallery_pids, gallery_cams, max_rank=50):
    all_ap = []
    all_cmc = []
    for index in range(distmat.shape[0]):
        order = np.argsort(distmat[index])
        valid = ~((gallery_pids[order] == query_pids[index]) & (gallery_cams[order] == query_cams[index]))
        matches = (gallery_pids[order][valid] == query_pids[index]).astype(np.int32)
        if matches.sum() == 0:
            continue
        cmc = matches.cumsum()
        cmc = (cmc > 0).astype(np.float32)
        cmc_vec = np.zeros(max_rank, dtype=np.float32)
        upto = min(max_rank, len(cmc))
        cmc_vec[:upto] = cmc[:upto]
        all_cmc.append(cmc_vec)
        cumulative_true_positives = matches.cumsum()
        precision = cumulative_true_positives / (np.arange(len(matches)) + 1)
        all_ap.append((precision * matches).sum() / matches.sum())
    mean_ap = float(np.mean(all_ap)) if all_ap else 0.0
    cmc_curve = np.mean(all_cmc, axis=0) if all_cmc else np.zeros(max_rank, dtype=np.float32)
    return mean_ap, cmc_curve


def evaluate_model(model, query_loader, gallery_loader, device):
    query_features, query_pids, query_cams = extract_features(model, query_loader, device)
    gallery_features, gallery_pids, gallery_cams = extract_features(model, gallery_loader, device)
    base_dist = 1.0 - np.matmul(query_features, gallery_features.T)
    map_value, cmc = eval_market_style(base_dist, query_pids, query_cams, gallery_pids, gallery_cams)
    rr_dist = re_ranking(query_features, gallery_features, k1=20, k2=6, lambda_value=0.3)
    map_rr, cmc_rr = eval_market_style(rr_dist, query_pids, query_cams, gallery_pids, gallery_cams)
    return {
        "mAP": float(map_value),
        "rank1": float(cmc[0]),
        "rank5": float(cmc[4]) if len(cmc) > 4 else float(cmc[-1]),
        "rank10": float(cmc[9]) if len(cmc) > 9 else float(cmc[-1]),
        "mAP_rr": float(map_rr),
        "rank1_rr": float(cmc_rr[0]),
        "query_count": int(len(query_pids)),
        "gallery_count": int(len(gallery_pids)),
    }


def save_training_curves(history, output_path):
    if not history:
        return
    epochs = [item["epoch"] for item in history]
    train_loss = [item["train_loss"] for item in history]
    map_values = [item.get("mAP") for item in history]
    map_rr_values = [item.get("mAP_rr") for item in history]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(epochs, train_loss, label="train_loss", color="#c44e52")
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)

    eval_epochs = [epochs[index] for index, value in enumerate(map_values) if value is not None]
    eval_map = [value for value in map_values if value is not None]
    eval_map_rr = [value for value in map_rr_values if value is not None]
    if eval_epochs:
        axes[1].plot(eval_epochs, eval_map, label="mAP", color="#4c72b0")
        axes[1].plot(eval_epochs, eval_map_rr, label="mAP_rr", color="#55a868")
    axes[1].set_title("Validation Metrics")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(fig)


def main():
    sys.stdout = TeeWriter(sys.stdout, LOG_HANDLE)
    sys.stderr = TeeWriter(sys.stderr, LOG_HANDLE)
    atexit.register(LOG_HANDLE.close)

    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    print(f"[debug] Device: {device}")
    if torch.cuda.is_available():
        print(f"[debug] GPU: {torch.cuda.get_device_name(0)}")
    print(f"[debug] PyTorch: {torch.__version__}")

    kaggle_input_root = Path("/kaggle/input")
    weights_dataset_root = kaggle_input_root / "mtmc-weights"
    warmstart_checkpoint = resolve_checkpoint_path(kaggle_input_root, "/kaggle/input/mtmc-weights/reid/resnet101ibn_cityflowv2_384px_best.pth")
    raw_root = resolve_cityflow_train_root(kaggle_input_root)

    cfg = {
        "dataset_root": "/kaggle/working/cityflowv2_reid",
        "crop_root": "/kaggle/working/cityflowv2_crops",
        "weights_output": "/kaggle/working/resnet101ibn_arcface_cityflowv2_best.pth",
        "checkpoint_dir": "/kaggle/working/checkpoints_arcface",
        "history_output": "/kaggle/working/training_history_resnet101ibn_arcface.json",
        "metrics_output": "/kaggle/working/final_metrics_resnet101ibn_arcface.json",
        "curves_output": "/kaggle/working/training_curves_resnet101ibn_arcface.png",
        "epochs": 160,
        "batch_size": 64,
        "num_identities": 16,
        "num_instances": 4,
        "eval_batch_size": 64,
        "lr": 3.5e-4,
        "warmup_epochs": 10,
        "warmup_factor": 0.01,
        "weight_decay": 5e-4,
        "label_smoothing": 0.1,
        "triplet_margin": 0.3,
        "arcface_s": 30.0,
        "arcface_m": 0.35,
        "center_weight": 5e-4,
        "center_start_epoch": 20,
        "center_lr": 0.5,
        "random_erasing_prob": 0.5,
        "img_size": [384, 384],
        "fp16": True,
        "eval_every": 10,
        "milestone_epochs": [80, 120],
        "gamma": 0.1,
        "gem_p": 3.0,
        "last_stride": 1,
        "gradient_clip": 5.0,
        "warmstart_checkpoint": warmstart_checkpoint,
        "use_cityflow_warmstart": warmstart_checkpoint is not None,
        "train_split_ratio": 0.7,
        "rerank_k1": 20,
        "rerank_k2": 6,
        "rerank_lambda": 0.3,
    }

    print(json.dumps(cfg, indent=2))
    os.makedirs(cfg["checkpoint_dir"], exist_ok=True)
    reid_root = Path(cfg["dataset_root"])
    extract_cityflow_crops(raw_root, reid_root)
    train_data, query_data, gallery_data, num_classes, num_cameras = parse_cityflowv2(cfg["dataset_root"])
    print(f"[debug] Train images: {len(train_data)} | classes: {num_classes} | cameras: {num_cameras}")
    print(f"[debug] Query images: {len(query_data)} | gallery images: {len(gallery_data)}")

    height, width = cfg["img_size"]
    train_transform = T.Compose([
        T.Resize((height, width), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(p=0.5),
        T.Pad(10),
        T.RandomCrop((height, width)),
        T.ColorJitter(brightness=0.2, contrast=0.15, saturation=0.1, hue=0.05),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        T.RandomErasing(p=cfg["random_erasing_prob"], value="random"),
    ])
    test_transform = T.Compose([
        T.Resize((height, width), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    assert cfg["batch_size"] == cfg["num_identities"] * cfg["num_instances"], "batch_size must equal P*K"
    train_loader = DataLoader(
        ReIDDataset(train_data, train_transform),
        batch_size=cfg["batch_size"],
        sampler=PKSampler(train_data, p=cfg["num_identities"], k=cfg["num_instances"]),
        num_workers=4,
        pin_memory=True,
        drop_last=True,
    )
    query_loader = DataLoader(
        ReIDDataset(query_data, test_transform),
        batch_size=cfg["eval_batch_size"],
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )
    gallery_loader = DataLoader(
        ReIDDataset(gallery_data, test_transform),
        batch_size=cfg["eval_batch_size"],
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )

    model = ReIDModelResNet101IBNArcFace(
        num_classes=num_classes,
        feat_dim=2048,
        last_stride=cfg["last_stride"],
        gem_p=cfg["gem_p"],
        arcface_s=cfg["arcface_s"],
        arcface_m=cfg["arcface_m"],
        pretrained=True,
    ).to(device)
    if torch.cuda.device_count() > 1:
        print(f"[debug] Using DataParallel across {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)

    if cfg["use_cityflow_warmstart"]:
        print(f"[debug] Loading 09d warm-start checkpoint from {cfg['warmstart_checkpoint']}")
        checkpoint = torch.load(cfg["warmstart_checkpoint"], map_location="cpu", weights_only=False)
        if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
            load_matching_state_dict(model, checkpoint["state_dict"])
        elif isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
            load_matching_state_dict(model, checkpoint["model_state_dict"])
        else:
            load_matching_state_dict(model, checkpoint)
    else:
        print("[debug] No 09d warm-start checkpoint found; training from ImageNet initialization only")

    criterion_id = CrossEntropyLabelSmooth(num_classes, epsilon=cfg["label_smoothing"])
    criterion_triplet = TripletLoss(margin=cfg["triplet_margin"])
    criterion_center = CenterLoss(num_classes, 2048).to(device)

    model_for_optim = unwrap_model(model)
    optimizer = torch.optim.AdamW([
        {"params": model_for_optim.conv1.parameters()},
        {"params": model_for_optim.bn1.parameters()},
        {"params": model_for_optim.layer1.parameters()},
        {"params": model_for_optim.layer2.parameters()},
        {"params": model_for_optim.layer3.parameters()},
        {"params": model_for_optim.layer4.parameters()},
        {"params": model_for_optim.pool.parameters()},
        {"params": model_for_optim.bottleneck.parameters()},
        {"params": model_for_optim.arcface_head.parameters()},
    ], lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    center_optimizer = torch.optim.SGD(criterion_center.parameters(), lr=cfg["center_lr"])
    warmup_scheduler = LinearLR(optimizer, start_factor=cfg["warmup_factor"], total_iters=cfg["warmup_epochs"])
    multistep_scheduler = MultiStepLR(optimizer, milestones=cfg["milestone_epochs"], gamma=cfg["gamma"])
    scheduler = SequentialLR(optimizer, [warmup_scheduler, multistep_scheduler], milestones=[cfg["warmup_epochs"]])
    scaler = torch.amp.GradScaler("cuda") if cfg["fp16"] and torch.cuda.is_available() else None

    best_metrics = {
        "mAP": 0.0,
        "rank1": 0.0,
        "mAP_rr": 0.0,
        "rank1_rr": 0.0,
        "epoch": -1,
    }
    history = []

    for epoch in range(cfg["epochs"]):
        model.train()
        running = {"loss": 0.0, "id": 0.0, "tri": 0.0, "cen": 0.0, "n": 0}
        epoch_start = time.time()
        optimizer.zero_grad(set_to_none=True)
        center_optimizer.zero_grad(set_to_none=True)

        for images, pids, cams, _ in train_loader:
            images = images.to(device)
            pids = pids.to(device).long()

            with torch.amp.autocast("cuda", enabled=scaler is not None):
                cls_score, global_feat, bn_feat = model(images, pids)
                loss_id = criterion_id(cls_score, pids)
                loss_tri = criterion_triplet(global_feat, pids)
                if (epoch + 1) >= cfg["center_start_epoch"]:
                    loss_cen = criterion_center(global_feat, pids) * cfg["center_weight"]
                else:
                    loss_cen = torch.tensor(0.0, device=device)
                loss = loss_id + loss_tri + loss_cen

            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg["gradient_clip"])
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg["gradient_clip"])
                optimizer.step()

            if (epoch + 1) >= cfg["center_start_epoch"]:
                for parameter in criterion_center.parameters():
                    if parameter.grad is not None:
                        parameter.grad.data *= 1.0 / cfg["center_weight"]
                center_optimizer.step()

            optimizer.zero_grad(set_to_none=True)
            center_optimizer.zero_grad(set_to_none=True)
            running["loss"] += float(loss.item())
            running["id"] += float(loss_id.item())
            running["tri"] += float(loss_tri.item())
            running["cen"] += float(loss_cen.item())
            running["n"] += 1

        scheduler.step()
        batch_count = max(running["n"], 1)
        epoch_record = {
            "epoch": epoch + 1,
            "lr": float(optimizer.param_groups[0]["lr"]),
            "train_loss": running["loss"] / batch_count,
            "loss_id": running["id"] / batch_count,
            "loss_tri": running["tri"] / batch_count,
            "loss_cen": running["cen"] / batch_count,
            "mAP": None,
            "rank1": None,
            "mAP_rr": None,
            "rank1_rr": None,
        }
        history.append(epoch_record)
        elapsed = time.time() - epoch_start
        print(
            f"E{epoch + 1:03d} | {elapsed:.0f}s | LR={epoch_record['lr']:.6f} | "
            f"L={epoch_record['train_loss']:.4f} ID={epoch_record['loss_id']:.4f} "
            f"Tri={epoch_record['loss_tri']:.4f} Cen={epoch_record['loss_cen']:.4f}"
        )

        if (epoch + 1) % cfg["eval_every"] == 0 or (epoch + 1) == cfg["epochs"]:
            metrics = evaluate_model(model, query_loader, gallery_loader, device)
            epoch_record.update(metrics)
            print(
                f"  >> mAP={metrics['mAP']:.4f}, R1={metrics['rank1']:.4f}, "
                f"mAP_rr={metrics['mAP_rr']:.4f}, R1_rr={metrics['rank1_rr']:.4f}"
            )
            checkpoint_payload = {
                "state_dict": get_model_state_dict(model),
                "epoch": epoch + 1,
                "mAP": metrics["mAP"],
                "rank1": metrics["rank1"],
                "mAP_rr": metrics["mAP_rr"],
                "rank1_rr": metrics["rank1_rr"],
                "config": cfg,
                "architecture": {
                    "backbone": "resnet101_ibn_a",
                    "pooling": "GeM",
                    "neck": "BNNeck",
                    "head": "ArcFaceHead",
                    "feat_dim": 2048,
                    "img_size": cfg["img_size"],
                },
                "history_tail": history[-5:],
            }
            torch.save(checkpoint_payload, f"{cfg['checkpoint_dir']}/ckpt_e{epoch + 1:03d}.pth")
            if metrics["mAP"] > best_metrics["mAP"]:
                best_metrics = {
                    "mAP": metrics["mAP"],
                    "rank1": metrics["rank1"],
                    "mAP_rr": metrics["mAP_rr"],
                    "rank1_rr": metrics["rank1_rr"],
                    "epoch": epoch + 1,
                }
                torch.save(checkpoint_payload, cfg["weights_output"])
                print(f"  * New best checkpoint saved at epoch {epoch + 1} with mAP={metrics['mAP']:.4f}")

        with open(cfg["history_output"], "w", encoding="utf-8") as handle:
            json.dump({"config": cfg, "history": history, "best": best_metrics}, handle, indent=2)

    save_training_curves(history, cfg["curves_output"])
    print(f"[debug] Training curves written to {cfg['curves_output']}")

    if not os.path.exists(cfg["weights_output"]):
        raise RuntimeError("No best checkpoint was saved during training")

    best_checkpoint = torch.load(cfg["weights_output"], map_location="cpu", weights_only=False)
    load_matching_state_dict(model, best_checkpoint["state_dict"])
    model.to(device)
    final_metrics = evaluate_model(model, query_loader, gallery_loader, device)
    final_summary = {
        "best_epoch": int(best_checkpoint["epoch"]),
        "mAP": float(final_metrics["mAP"]),
        "rank1": float(final_metrics["rank1"]),
        "rank5": float(final_metrics["rank5"]),
        "rank10": float(final_metrics["rank10"]),
        "mAP_rr": float(final_metrics["mAP_rr"]),
        "rank1_rr": float(final_metrics["rank1_rr"]),
        "config": cfg,
        "architecture": best_checkpoint["architecture"],
    }
    with open(cfg["metrics_output"], "w", encoding="utf-8") as handle:
        json.dump(final_summary, handle, indent=2)

    print("FINAL METRICS")
    print(json.dumps(final_summary, indent=2))

    checkpoint_keys = list(best_checkpoint["state_dict"].keys())
    assert any(key.startswith("conv1.") for key in checkpoint_keys), "Missing conv1 weights"
    assert any(key.startswith("layer1.") for key in checkpoint_keys), "Missing layer1 weights"
    assert any(key.startswith("pool.") for key in checkpoint_keys), "Missing GeM weights"
    assert any(key.startswith("bottleneck.") for key in checkpoint_keys), "Missing BNNeck weights"
    assert any(key.startswith("arcface_head.") for key in checkpoint_keys), "Missing ArcFace head weights"
    print("[debug] Checkpoint structure validated")


if __name__ == "__main__":
    main()
'''


script_path = "/kaggle/working/train_09i_resnet101ibn_arcface.py"
with open(script_path, "w", encoding="utf-8") as handle:
    handle.write(script_content)

print(f"Training script written to {script_path}")
print(f"Script size: {os.path.getsize(script_path) / 1024:.1f} KB")

In [ ]:
import subprocess
import sys


result = subprocess.run(
    [sys.executable, "/kaggle/working/train_09i_resnet101ibn_arcface.py"],
    check=False,
    text=True,
)
print(f"Training process exited with code: {result.returncode}")
if result.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {result.returncode}")